In [34]:
from z3 import *

#TODO change so that we can call name,color as member without parentheses
#TODO if number of solutions is greater than 1 then find unnessaserry condition so that after its elimination array of solutions will be same 

def IsPermutationWithDistinctPrecond(a,b):
    return And([ Or([ a[i]==b[j] for i in range(len(a)) ])  for j in range(len(b)) ])

def create_game(game):
    s=Solver()
    Var = Datatype('Var')
    constr_params=[]
    size=0
    temp_vars=[]
    chars="abcdefghijklmnopqrstuvwxyz"
    for name,values in game.items():
        size=len(values)
        if isinstance(values[0], str):
            constr_params.append((name+"_m",StringSort()))
        else:
            constr_params.append((name+"_m",RealSort()))
    #print(*constr_params)
    Var.declare("make",*constr_params) 
    Var = Var.create()
    vars = [Const("x%i" % i,Var) for i in range(size)]

    

    for i in range(size):
        tmp_var=Const("t%s" % chars[i] ,Var)
        globals()[chars[i]]=tmp_var# temp var for Exists
        #for name,values in game.items():
        #    setattr(tmp_var, name, lambda: getattr(Var, name)(tmp_var) )
        
        temp_vars.append(tmp_var)

    for name,values in game.items():
        setattr(temp_vars[0].__class__, name, (lambda self,name=name: getattr(Var, name+"_m")(self)))
        #print([getattr(Var, name+"_m")(var for var in vars ])

        var_values=[getattr(Var, name+"_m")(var) for var in vars ]
        s.add(Distinct(var_values))
        s.add(IsPermutationWithDistinctPrecond(var_values,values))
    return (s,vars,Var,temp_vars,game)

def get_new_conds(state,conds):
    (s,vars,Var,temp_vars,game) = state
    new_conds=[]
    for cond in conds:
        if isinstance(cond, tuple) and len(cond)==2:
            new_conds.append(And(Exists(temp_vars,And(And(*cond),Distinct(temp_vars),IsPermutationWithDistinctPrecond(temp_vars,vars))),
                             ForAll(temp_vars,Implies(And(Distinct(temp_vars),IsPermutationWithDistinctPrecond(temp_vars,vars)),    Implies(*cond)))
                             ))
        elif cond.is_exists():
            new_conds.append(Exists(temp_vars,And(IsPermutationWithDistinctPrecond(temp_vars,vars),cond.body())))
            #print(new_conds[-1])
        elif cond.is_forall():
            new_conds.append(ForAll(temp_vars,Implies(And(Distinct(temp_vars),IsPermutationWithDistinctPrecond(temp_vars,vars)),    cond.body())))
            #print(new_conds[-1])
        else:
            print("err")
            exit()
            new_conds.append(cond)
    return new_conds

def add_sol(state,sol):
    (s,vars,Var,temp_vars,game) = state
    if sol:
        solution=[]
        for row in sol:
            new_row=[StringVal(x) if isinstance(x, str) else x for x in row ]
            solution.append(Var.make(*new_row))
        s.add(IsPermutationWithDistinctPrecond(vars,solution))

def solve_game(state,conds,sol):
    (s,vars,Var,temp_vars,game) = state

    s.add(And(get_new_conds(state,conds)))

    number_of_solutions=0
    add_sol(state,sol)

    while True:
        st=s.check()
        if st==sat:
            number_of_solutions+=1
            m = s.model()

            for x in range(len(vars)):
                print(tuple([m.eval(getattr(Var, name+"_m")(vars[x])).as_string() for name in game.keys()]))
                #for name in game.keys():
                #    print(name+"="+m.eval(getattr(Var, name+"_m")(vars[x])).as_string(),end=' ')
                #print()
            print()
            #break
            s.add(Not(IsPermutationWithDistinctPrecond([m.eval(var)  for var in vars ],vars)))
        else:
            break

    if number_of_solutions==0:
        print("not able to find solution")
    elif number_of_solutions>1:
        print("unexpected number of solutions: "+str(number_of_solutions))
    return number_of_solutions

def get_eliminated_conds(should_be_eliminated_array,conds):
    eliminated_conds=[]
    for i in range(len(conds)):
        should_be_eliminated=should_be_eliminated_array[i]
        cond=conds[i]
        eliminated_conds.append(Implies(Not(should_be_eliminated),cond==True))
    return eliminated_conds

def find_wrong_cond(state,conds,sol):

    (s,vars,Var,temp_vars,game) = state
    (s,vars,Var,temp_vars,game) = create_game(game)

    conds = get_new_conds(state,conds)
    should_be_eliminated_array=[Const("el%i" % i,BoolSort()) for i in range(len(conds))]
    temp_should_be_eliminated_array=[Const("temp_el%i" % i,BoolSort()) for i in range(len(conds))]
    s.add(And(get_eliminated_conds(should_be_eliminated_array,conds)))
    s.add(Not(Exists(temp_should_be_eliminated_array,And(#minimizing eliminated conditions
        Sum(temp_should_be_eliminated_array)<Sum(should_be_eliminated_array),
        And(get_eliminated_conds(temp_should_be_eliminated_array,conds))
    ))))
    add_sol(state,sol)
    at_least_one_found=False
    while True:
        st=s.check()
        if st==sat:
            if at_least_one_found==False:
                print("sets of possible wrong conditions: ",end='')
            at_least_one_found=True
            m = s.model()
            print([i for i in range(len(should_be_eliminated_array)) if m.eval(should_be_eliminated_array[i]) ],end=' ')
            s.add(Not(And([m.eval(should_be_eliminated_array[i])==should_be_eliminated_array[i] for i in range(len(should_be_eliminated_array)) ])))
        else:
            break
    if at_least_one_found==False:
        print("not able to find wrong conditions")



In [50]:
from z3lib import *
#https://www.braingle.com/brainteasers/logicgrid/index.php?id=44985
game={
    "name": ["A","D","J","L","S"],  
    "age": [28,29,30,31,32],
    "mon": [2,5,6,8,10], #month
    "day": [9,11,15,29,31],
    "hour": [6,9,14,17,19],  

}
state = create_game(game)
conds=[
    #comman knowledge
    (a.mon()==2, a.day()<=29),
    (a.mon()==5, a.day()<=31),
    (a.mon()==6, a.day()<=30),
    (a.mon()==8, a.day()<=31),
    (a.mon()==10, a.day()<=31),

    # Two of the boys were born in a leap year.      
    #NOTE its mean that if some of boys born in 2/29 then there are other boy who born in leap year
    #NOTE if no boy born in 2/29 then we can conclude that there are at least two boys whose age difference is 4
    # ForAll([a],Implies(And(a.mon()==2,a.day()==29,a.name()!="D",a.name()!="L"),        Exists([b],  And(a.age()%4==b.age()%4,
    #                                                                                                     b.name()!="D",b.name()!="L"))                                                           
    # ))),
    
    # Neither girl was born in the morning.
    (Or(a.name()=="D",a.name()=="L"),           a.hour()>12),
    
    # Both girls were born in the first half of the month. 
    (Or(a.name()=="D",a.name()=="L"),           a.day()<=15),

    # One of the boys was born three hours later in the day than one of the girls.
    Exists([a,b],And(Or(a.name()=="D",a.name()=="L"),And(b.name()!="D",b.name()!="L"),a.hour()+3==b.hour())),

    #No one celebrated a golden birthday last year, and no one will celebrate a golden birthday this year.
    ForAll([a],And(a.age()!=a.day(),a.age()-1!=a.day())),

    #The five people are: Alvin, the person who is 32 years old, the person born in August, the person born on the 11th, and the person born at 9:00 AM.
    (And(a.name()=="A",b.age()==32,c.mon()==8,d.day()==11,e.hour()==9),           Distinct(a,b,c,d,e)),

    #The person born at 6:00 AM is older than the girl born on the 15th, who is older than the boy born in October.
    (And(a.hour()==6,b.day()==15,Or(b.name()=="D",b.name()=="L"),c.mon()==10,And(c.name()!="D",c.name()!="L")),           And(c.age()<b.age(),b.age()<a.age())),

    #Jerry was born earlier in the year than the person born on the 11th, who was born earlier in the year than the person who is 28 years old.
    (And(a.name()=="J",b.day()==15,c.age()==28),           And(a.mon()<b.mon(),b.mon()<c.mon())),

    #The person born in May was born earlier in the month than the person born at 7:00 PM, who was born earlier in the month than the person who is 30 years old.
    (And(a.mon()==5,b.hour()==19,c.age()==30),           And(a.day()<b.day(),b.day()<c.day())),

    #The person born on the 29th was born earlier in the day than the person who is 31 years old, who was born earlier in the day than the person born in June.
    (And(a.day()==29,b.age()==31,c.mon()==6),           And(a.hour()<b.hour(),b.hour()<c.hour())),

    #Steve and Donna have the two closest birthdays. 
    #NOTE note sure what he mean by birthday, maybe how long it take from celebration of birthday D from S 
    #TODO not exists c,d so that Abs(c.day()-d.day())*24+Abs(c.hour()-d.hour()) < Abs(a.day()-b.day())*24+Abs(a.hour()-b.hour())
    (And(a.name()=="D",b.name()=="S"),  Abs(a.day()-b.day())==2   ),
    
]

sol=[
("A", 28 , 10, 31, 17),
("D", 29 , 6, 11, 19),
("J", 32 , 2, 29, 6),
("L", 30 , 8, 15, 14),
("S", 31 , 5, 9, 9)
]
sol=[]
number_of_solutions = solve_game(state,conds,sol)
if number_of_solutions==0:
    find_wrong_cond(state,conds,sol)

('L', '30', '8', '15', '14')
('D', '29', '6', '11', '19')
('J', '32', '2', '29', '6')
('S', '31', '5', '9', '9')
('A', '28', '10', '31', '17')



In [10]:
#https://logic.puzzlebaron.com/play.php?u2=f0eb13969cfa47ff7f4f2074f11b923f
game={
    "name": ["E","H","R","V"],  
    "dist": [15,25,35,45],  
    "color": ["b","g","s","y"],
}

state = create_game(game)

conds=[#all Objects satisfy each of conditions and exists at least one Object which satisfy 
    (And(a.name()=="H",b.name()=="R"),       a.dist()==b.dist()-10), #Henrietta's design landed 10 feet shorter than Roderick's design.
    (And(a.name()=="V"),                     a.color()=="s"),        #Valerie's design was silver.
    (And(a.color()=="g", b.name()=="V"),     a.dist()==b.dist()-10), #The green plane landed 10 feet shorter than Valerie's design.
    (And(a.dist()==45),                      a.color()=="b"),        #The airplane that went 45 feet was blue.
    #(And(a.dist()==45),                      a.color()=="s"),        #wrong alternative of above one
    (And(a.color()=="g", b.color()=="y"),    a.dist()<b.dist())      #The green plane landed somewhat shorter than the yellow plane.
]

sol=[
    ("H",35,"y"),
    ("E",15,"g"),
    ("V",25,"s"),
    ("R",45,"b")
]

number_of_solutions = solve_game(state,conds)
if number_of_solutions==0:
    find_wrong_cond(state,conds,sol)


('V', '25', 's')
('R', '45', 'b')
('E', '15', 'g')
('H', '35', 'y')



In [11]:
#https://logic.puzzlebaron.com/play.php?u2=9096196665d016173f355cb0f383d64c
game={
    "name": ["C","H","T","W"],  
    "time": [9,10,11,12],  
    "dis": ["b","f","h","v"], #disease
}
state = create_game(game)
conds=[
    And(a.name()=="W",Or(a.time()=="9", a.dis()=="b")),# Willard is either the person with the 9:00am appointment or the person suffering from back pain.
    And(a.dis()=="f",b.name()=="C", a.time()==b.time()-1),# The person suffering from foot pain has an appointment 1 hour before Chester.
    And(a.name()=="H", b.name()=="C", a.time()==b.time()+1),# Hugh has an appointment 1 hour after Chester.
    And(a.name()=="T", b.dis()=="v",Or(And(a.time()==9,b.time()==11),And(b.time()==9,a.time()==11))),# Of Terry and the patient suffering from vertigo, one has the 9:00am appointment and the other has the 11:00am appointment.
]
conds=[#all Objects satisfy each of conditions and exists at least one Object which satisfy 
    (And(a.name()=="W"),                 Or(a.time()=="9", a.dis()=="b")),# Willard is either the person with the 9:00am appointment or the person suffering from back pain.
    (And(a.dis()=="f",b.name()=="C"),    a.time()==b.time()-1),# The person suffering from foot pain has an appointment 1 hour before Chester.
    (And(a.name()=="H", b.name()=="C"),  a.time()==b.time()+1),# Hugh has an appointment 1 hour after Chester.
    (And(a.name()=="T", b.dis()=="v"),   Or(And(a.time()==9,b.time()==11),And(b.time()==9,a.time()==11))),# Of Terry and the patient suffering from vertigo, one has the 9:00am appointment and the other has the 11:00am appointment.
]
sol=[

]
number_of_solutions = solve_game(state,conds)
if number_of_solutions==0:
    find_wrong_cond(state,conds,sol)

('H', '11', 'v')
('W', '12', 'b')
('T', '9', 'f')
('C', '10', 'h')

